# 🎵 TuneWave Music Player — Master Integrated Core
### Complete Implementation of Milestones 1, 2, and 3
**Data Structures:** Doubly Linked List • Hash Map • FIFO Queue • Bounded LIFO Stack • Binary Search Tree • Undirected Graph

---

## 🌟 Executive Overview & Deliverable Summary
This notebook demonstrates the fully unified, production-grade **TuneWave Music Player Core**, successfully fulfilling all requirements, invariants, and constraints across all 3 project milestones:

| Milestone | Target System | Core Data Structures | Key Invariants & Rules |
| :--- | :--- | :--- | :--- |
| **Milestone 1** | **Playlist Management** | Doubly Linked List + Hash Map | Track uniqueness in $O(1)$; repeat wrapping vs. terminal stop; bidirectional $O(1)$ navigation; dynamic $O(1)$ deletion. |
| **Milestone 2** | **Up Next & History** | FIFO Queue + Bounded LIFO Stack | Up Next preemption; playlist cursor immutability; 10-song bounded history with $O(1)$ oldest drop; `back()` rewinds without moving playlist. |
| **Milestone 3** | **Search & Recommendations** | Binary Search Tree + Undirected Graph | BST title indexing; pruned prefix search skipping unmatchable subtrees; 2-hop artist recommendations excluding direct links. |


## 🏗️ Integrated Architecture Diagram

```
                                +---------------------------------------------+
                                |            TuneWave Core Engine             |
                                +---------------------------------------------+
                                                       |
          +--------------------------------------------+--------------------------------------------+
          |                                            |                                            |
          v                                            v                                            v
+-------------------+                        +-------------------+                        +-------------------+
|    Milestone 1    |                        |    Milestone 2    |                        |    Milestone 3    |
|     Playlist      |                        |  Up Next & Hist   |                        |  Search & Recs    |
+-------------------+                        +-------------------+                        +-------------------+
| • Doubly Linked   |                        | • FIFO Queue      |                        | • Binary Search   |
|   List (DLL)      |                        |   (Up Next)       |                        |   Tree (BST)      |
| • Hash Map (O(1)  |                        | • Bounded Stack   |                        | • Undirected Graph|
|   Uniqueness &    |                        |   (Max 10 LIFO)   |                        |   (2-Hop Recs)    |
|   Deletions)      |                        | • Frozen Cursor   |                        | • Subtree Pruning |
+-------------------+                        +-------------------+                        +-------------------+
```


In [1]:
# Import the fully integrated TuneWave engine
from tunewave_player import Song, SongNode, Playlist, UpNextQueue, PlayHistory, SongSearchBST, ArtistGraph, TuneWavePlayer

print("✓ All TuneWave core components loaded successfully!")


✓ All TuneWave core components loaded successfully!


## 🧩 Milestone 1 Walkthrough: Playlist (DLL + Hash Map)
- A song can appear only once in a playlist (Hash map check in $O(1)$).
- Bidirectional navigation with `next()` and `previous()`.
- Terminal behavior: with repeat OFF, `next()` stays on the last song and returns `'end'`; with repeat ON, wraps to the first song.
- Dynamic deletion in $O(1)$ time while playing.


In [2]:
# 1. Initialize Playlist and populate songs
playlist = Playlist("Indie & Synthwave")

s1 = Song("S1", "Midnight City", "M83", 244)
s2 = Song("S2", "Resonance", "HOME", 212)
s3 = Song("S3", "Nightcall", "Kavinsky", 259)

print("Adding S1:", playlist.addSong(s1))
print("Adding S2:", playlist.addSong(s2))
print("Adding S3:", playlist.addSong(s3))

# Verify Uniqueness Rule:
duplicate_attempt = playlist.addSong(s1)
print("\nDuplicate addition attempt of S1:", duplicate_attempt)
assert duplicate_attempt is False, "Duplicate must be rejected!"
print("✓ Duplicate prevented in O(1) time without scanning!")

# 2. Test Playback and Repeat OFF vs ON
print("\n-- Playback with Repeat OFF --")
print("Initial track:", playlist.getCurrentSong())
print("1. next():", playlist.next())
print("2. next():", playlist.next())
print("3. next() at end of playlist:", playlist.next())  # Rule: stays on last song and returns 'end'
print("Active track retained at end:", playlist.getCurrentSong())

print("\n-- Toggling Repeat ON --")
playlist.toggleRepeat()
print("Repeat enabled:", playlist.repeat)
print("next() with repeat ON (should wrap to first track):", playlist.next())
print("Active track wrapped to:", playlist.getCurrentSong())

# 3. Dynamic Deletion
print("\n-- Dynamic Deletion in O(1) --")
print("Removing active track 'Midnight City' [S1]:", playlist.removeSong("S1"))
print("Current track automatically reassigned to:", playlist.getCurrentSong())
assert "S1" not in playlist.song_map
print("✓ Track unlinked and removed from hash map in O(1)!")


Adding S1: True
Adding S2: True
Adding S3: True

Duplicate addition attempt of S1: False
✓ Duplicate prevented in O(1) time without scanning!

-- Playback with Repeat OFF --
Initial track: 'Midnight City' by M83 [S1]
1. next(): 'Resonance' by HOME [S2]
2. next(): 'Nightcall' by Kavinsky [S3]
3. next() at end of playlist: end
Active track retained at end: 'Nightcall' by Kavinsky [S3]

-- Toggling Repeat ON --
Repeat enabled: True
next() with repeat ON (should wrap to first track): 'Midnight City' by M83 [S1]
Active track wrapped to: 'Midnight City' by M83 [S1]

-- Dynamic Deletion in O(1) --
Removing active track 'Midnight City' [S1]: True
Current track automatically reassigned to: 'Resonance' by HOME [S2]
✓ Track unlinked and removed from hash map in O(1)!


## 🧩 Milestone 2 Walkthrough: Up Next & Play History (Queue + Stack)
- Songs in the Up Next queue play **before** the playlist continues.
- **Critical Invariant:** Playing from Up Next or history **does not move the playlist position**.
- History keeps the **last 10 songs played** and **drops the oldest** in $O(1)$.
- `back()` replays the top song and removes it from history without altering playlist position.


In [3]:
# Setup player with playlist
album = Playlist("Synth Legends")
t1 = Song("T1", "A Real Hero", "College & Electric Youth")
t2 = Song("T2", "Tech Noir", "Gunship")
t3 = Song("T3", "Turbo Killer", "Carpenter Brut")
album.addSong(t1)
album.addSong(t2)
album.addSong(t3)

player = TuneWavePlayer("Synth Legends")
player.playlist = album
player.currently_playing = t1

# Register bonus tracks for Up Next
u1 = Song("U1", "Days of Thunder", "The Midnight")
u2 = Song("U2", "Vampires", "The Midnight")
player.catalog["U1"] = u1
player.catalog["U2"] = u2

print("Initial playing track:", player.currently_playing)
print("Playlist cursor:", player.playlist.getCurrentSong())

# Advance to T2
print("\n-- Advancing to T2 --")
print("Now playing:", player.playNext())
print("Playlist cursor:", player.playlist.getCurrentSong())

# Queue 2 tracks into Up Next
print("\n-- Enqueueing U1 and U2 into Up Next --")
player.addToUpNext("U1")
player.addToUpNext("U2")
print("Up Next queue count:", len(player.up_next))

# Play Next -> Must play U1, keeping playlist cursor frozen on T2!
print("\n-- Calling playNext() (Up Next takes precedence!) --")
played_song = player.playNext()
print("Played:", played_song)
print("Playlist cursor (MUST STILL BE T2):", player.playlist.getCurrentSong())
assert player.playlist.getCurrentSong() == t2, "Playlist cursor must NOT move when playing from Up Next!"

# Play Next -> Must play U2, keeping playlist cursor frozen on T2!
print("\n-- Calling playNext() (Playing second Up Next song) --")
played_song2 = player.playNext()
print("Played:", played_song2)
print("Playlist cursor (MUST STILL BE T2):", player.playlist.getCurrentSong())
assert player.playlist.getCurrentSong() == t2

# Play Next -> Up Next is exhausted! Resumes playlist at T3!
print("\n-- Calling playNext() (Up Next empty -> resumes playlist) --")
resumed_song = player.playNext()
print("Played:", resumed_song)
print("Playlist cursor:", player.playlist.getCurrentSong())
assert resumed_song == t3

# Test back()
print("\n-- Testing back() rewinds history without moving playlist cursor --")
print("1st back():", player.back())  # Replays U2
print("Playlist cursor after back():", player.playlist.getCurrentSong())
assert player.playlist.getCurrentSong() == t3

print("2nd back():", player.back())  # Replays U1
print("Playlist cursor after back():", player.playlist.getCurrentSong())
assert player.playlist.getCurrentSong() == t3

print("✓ Milestone 2 Priority, History & Cursor Invariants 100% verified!")


Initial playing track: 'A Real Hero' by College & Electric Youth [T1]
Playlist cursor: 'A Real Hero' by College & Electric Youth [T1]

-- Advancing to T2 --
Now playing: 'Tech Noir' by Gunship [T2]
Playlist cursor: 'Tech Noir' by Gunship [T2]

-- Enqueueing U1 and U2 into Up Next --
Up Next queue count: 2

-- Calling playNext() (Up Next takes precedence!) --
Played: 'Days of Thunder' by The Midnight [U1]
Playlist cursor (MUST STILL BE T2): 'Tech Noir' by Gunship [T2]

-- Calling playNext() (Playing second Up Next song) --
Played: 'Vampires' by The Midnight [U2]
Playlist cursor (MUST STILL BE T2): 'Tech Noir' by Gunship [T2]

-- Calling playNext() (Up Next empty -> resumes playlist) --
Played: 'Turbo Killer' by Carpenter Brut [T3]
Playlist cursor: 'Turbo Killer' by Carpenter Brut [T3]

-- Testing back() rewinds history without moving playlist cursor --
1st back(): 'Vampires' by The Midnight [U2]
Playlist cursor after back(): 'Turbo Killer' by Carpenter Brut [T3]
2nd back(): 'Days of Thu

### 📦 History Invariant Verification: Bounded to Last 10 Songs
Demonstrating that the 11th played song causes the oldest song to be dropped in $O(1)$.


In [4]:
history_stack = PlayHistory(capacity=10)

# Simulate playing 15 tracks
for i in range(1, 16):
    s = Song(f"H{i}", f"Track {i}", "Artist")
    history_stack.push(s)

print(f"Total songs pushed: 15")
print(f"Stack size: {len(history_stack)} (Strictly <= 10)")
print("Surviving songs in history stack (oldest to newest):")
for idx, s in enumerate(history_stack.to_list(), 1):
    print(f"  {idx}. {s.title} [{s.song_id}]")

assert len(history_stack) == 10
assert history_stack.to_list()[0].song_id == "H6", "Oldest songs H1..H5 must be dropped!"
assert history_stack.peek().song_id == "H15", "Newest song must be on top of stack!"
print("✓ Bounded 10-song history and oldest eviction verified!")


Total songs pushed: 15
Stack size: 10 (Strictly <= 10)
Surviving songs in history stack (oldest to newest):
  1. Track 6 [H6]
  2. Track 7 [H7]
  3. Track 8 [H8]
  4. Track 9 [H9]
  5. Track 10 [H10]
  6. Track 11 [H11]
  7. Track 12 [H12]
  8. Track 13 [H13]
  9. Track 14 [H14]
  10. Track 15 [H15]
✓ Bounded 10-song history and oldest eviction verified!


## 🧩 Milestone 3 Walkthrough: Song Search & Artist Recommendations (BST + Graph)
- Songs stored in a **Binary Search Tree** keyed by `title`.
- `prefixSearch(prefix)` returns matching titles in **alphabetical order** and **skips subtrees that cannot match**.
- Artists form an **undirected graph** of related links.
- `recommendArtists(artist)` returns artists **exactly two links away** who are **not already directly related**, in **alphabetical order**.


In [5]:
# 1. Index Songs in BST
bst = SongSearchBST()
titles = [
    ("Hotel California", "S101"),
    ("Bohemian Rhapsody", "S102"),
    ("Stairway to Heaven", "S103"),
    ("Blinding Lights", "S104"),
    ("Circles", "S105"),
    ("Starboy", "S106"),
    ("Stay", "S107"),
    ("Bad Guy", "S108"),
    ("Save Your Tears", "S109")
]

for title, sid in titles:
    bst.addSong(title, sid)

print("BST indexed with", len(bst), "songs.")

# Exact Search
print("\n-- Exact Search --")
print("findSong('Starboy'):", bst.findSong("Starboy"))
print("findSong('Bad Guy'):", bst.findSong("Bad Guy"))
print("findSong('Non-Existent'):", bst.findSong("Non-Existent"))

# Pruned Prefix Search
print("\n-- Pruned Prefix Search (Alphabetical Order) --")
prefix_sta = bst.prefixSearch("Sta")
print("prefixSearch('Sta'):", prefix_sta)
assert prefix_sta == ["Stairway to Heaven", "Starboy", "Stay"], f"Got {prefix_sta}"
assert prefix_sta == sorted(prefix_sta), "Must be in alphabetical order!"

prefix_b = bst.prefixSearch("B")
print("prefixSearch('B'):", prefix_b)
assert prefix_b == ["Bad Guy", "Blinding Lights", "Bohemian Rhapsody"]
assert prefix_b == sorted(prefix_b)

# 2. Artist Graph & 2-Hop Recommendations
graph = ArtistGraph()

# Add Relations:
# The Weeknd <--> Daft Punk
# The Weeknd <--> Travis Scott
# Daft Punk <--> Pharrell Williams
# Daft Punk <--> Kanye West
# Travis Scott <--> Drake
# The Weeknd <--> Drake (DIRECT LINK!)

graph.addRelation("The Weeknd", "Daft Punk")
graph.addRelation("The Weeknd", "Travis Scott")
graph.addRelation("Daft Punk", "Pharrell Williams")
graph.addRelation("Daft Punk", "Kanye West")
graph.addRelation("Travis Scott", "Drake")
graph.addRelation("The Weeknd", "Drake")  # Direct link!

print("\n-- Artist Recommendations for 'The Weeknd' --")
weeknd_recs = graph.recommendArtists("The Weeknd")
print("Recommendations:", weeknd_recs)

# Drake is 2 hops away via Travis Scott, BUT Drake is also 1 hop away directly!
# Rule: Must NOT be already directly related!
assert "Drake" not in weeknd_recs, "Drake is directly linked and must be excluded!"
assert weeknd_recs == ["Kanye West", "Pharrell Williams"]
assert weeknd_recs == sorted(weeknd_recs), "Recommendations must be in alphabetical order!"

print("[OK] 2-hop recommendations, direct link exclusion, and alphabetical ordering verified!")


BST indexed with 9 songs.

-- Exact Search --
findSong('Starboy'): S106
findSong('Bad Guy'): S108
findSong('Non-Existent'): None

-- Pruned Prefix Search (Alphabetical Order) --
prefixSearch('Sta'): ['Stairway to Heaven', 'Starboy', 'Stay']
prefixSearch('B'): ['Bad Guy', 'Blinding Lights', 'Bohemian Rhapsody']

-- Artist Recommendations for 'The Weeknd' --
Recommendations: ['Kanye West', 'Pharrell Williams']
[OK] 2-hop recommendations, direct link exclusion, and alphabetical ordering verified!


## 🛡️ Master Automated Test Suite
Exhaustive verification of all milestones, constraints, and data structure invariants.


In [6]:
def run_master_test_suite():
    player = TuneWavePlayer("Master Suite")

    # Milestone 1: Playlist
    s1 = Song("S1", "Song A", "Artist 1")
    s2 = Song("S2", "Song B", "Artist 1")
    s3 = Song("S3", "Song C", "Artist 2")
    assert player.addSongToPlaylist(s1) is True
    assert player.addSongToPlaylist(s2) is True
    assert player.addSongToPlaylist(s3) is True
    assert player.addSongToPlaylist(s1) is False  # Duplicate
    assert len(player.playlist) == 3

    assert player.next() == s2
    assert player.next() == s3
    assert player.next() == "end"  # Repeat OFF stays at end
    player.toggleRepeat()
    assert player.next() == s1     # Repeat ON wraps to head
    assert player.previous() == s3 # Repeat ON wraps to tail
    assert player.removeSongFromPlaylist("S2") is True
    assert len(player.playlist) == 2

    # Milestone 2: Up Next & History
    u1 = Song("U1", "Extra Track 1", "Artist 3")
    u2 = Song("U2", "Extra Track 2", "Artist 4")
    player.catalog["U1"] = u1
    player.catalog["U2"] = u2
    assert player.addToUpNext("U1") is True
    assert player.addToUpNext("U2") is True

    curr_p = player.playlist.getCurrentSong()
    p_up1 = player.playNext()
    assert p_up1 == u1
    assert player.playlist.getCurrentSong() == curr_p, "Playlist cursor moved during Up Next!"
    p_up2 = player.playNext()
    assert p_up2 == u2
    assert player.playlist.getCurrentSong() == curr_p, "Playlist cursor moved during Up Next!"

    # Back operation: u2 is currently playing, so top of history is u1!
    assert player.back() == u1
    assert player.playlist.getCurrentSong() == curr_p, "Playlist cursor moved during back!"
    assert player.back() == s1
    assert player.playlist.getCurrentSong() == curr_p, "Playlist cursor moved during back!"

    # History 10-song bound
    for i in range(15):
        player.history.push(Song(f"TEST_{i}", f"T_{i}", "Art"))
    assert len(player.history) == 10
    assert player.history.to_list()[0].song_id == "TEST_5"

    # Milestone 3: Search & Graph
    player.addSong("Sunflower", "SF1")
    player.addSong("Sunroof", "SF2")
    player.addSong("Summer", "SF3")
    assert player.findSong("Sunflower") == "SF1"
    assert player.findSong("NotReal") is None
    prefix_res = player.prefixSearch("Sun")
    assert prefix_res == ["Sunflower", "Sunroof"]

    player.addRelation("ArtA", "ArtB")
    player.addRelation("ArtB", "ArtC")
    player.addRelation("ArtB", "ArtD")
    assert player.recommendArtists("ArtA") == ["ArtC", "ArtD"]
    player.addRelation("ArtA", "ArtC")  # Direct link
    assert player.recommendArtists("ArtA") == ["ArtD"]

    print("=====================================================================")
    print("🎉 ALL TESTS PASSED! TUNEWAVE PLAYER IS 100% OPERATIONAL & VERIFIED!")
    print("=====================================================================")

run_master_test_suite()


🎉 ALL TESTS PASSED! TUNEWAVE PLAYER IS 100% OPERATIONAL & VERIFIED!


## 📊 Final Complexity & Invariant Scorecard

$$egin{array}{|l|l|c|c|l|}
\hline
	extbf{Component} & 	extbf{Operation} & 	extbf{Time} & 	extbf{Space} & 	extbf{Invariants \& Rules Verified} \
\hline
	ext{Milestone 1} & 	ext{addSong(song)} & O(1) & O(1) & 	ext{Enforces unique track constraint via Hash Map} \
	ext{Milestone 1} & 	ext{removeSong(id)} & O(1) & O(1) & 	ext{O(1) Hash Map lookup + DLL pointer rewire} \
	ext{Milestone 1} & 	ext{next() / previous()} & O(1) & O(1) & 	ext{Repeat ON wraps; Repeat OFF returns 'end'} \
	ext{Milestone 1} & 	ext{toggleRepeat()} & O(1) & O(1) & 	ext{Atomic state inversion} \
\hline
	ext{Milestone 2} & 	ext{addToUpNext(id)} & O(1) & O(1) & 	ext{FIFO Queue priority buffer} \
	ext{Milestone 2} & 	ext{playNext()} & O(1) & O(1) & 	ext{Up Next preemption; playlist cursor frozen} \
	ext{Milestone 2} & 	ext{back()} & O(1) & O(1) & 	ext{LIFO stack pop; drops oldest at size 10} \
\hline
	ext{Milestone 3} & 	ext{addSong / findSong} & O(h) & O(1) & 	ext{BST keyed by title} \
	ext{Milestone 3} & 	ext{prefixSearch} & O(h + K) & O(K) & 	ext{Skips unmatchable subtrees; alphabetical} \
	ext{Milestone 3} & 	ext{recommendArtists} & O(N_1 \cdot ar{d}) & O(M) & 	ext{Exactly 2 links away; filters direct links} \
\hline
\end{array}$$
